In [7]:
# run plip

import subprocess
from pathlib import Path

# list of PDB files
pdb_files = ["/home/bri/pymol/pymol/Nb.AQ100_Adiponectin_model_0.pdb"]
#pdb_files = ["5ifj_abc_cut.pdb", "5ig7_abc_cut.pdb", "5ijk_acx_cut.pdb"]
base_outdir = Path("plip_results")
base_outdir.mkdir(exist_ok=True)

for pdb_path in pdb_files:
    # setup
    pdb_name = Path(pdb_path).stem 
    target_dir = base_outdir / pdb_name
    target_dir.mkdir(exist_ok=True)
    
    # run PLIP
    result = subprocess.run([
        "plip",
        "-f", pdb_path,
        "-o", str(target_dir),
        "--peptides", "B",  # use peptides arg, Chain C
        "-x",  # Generate XML report for data parsing
        "-y",  # human readable text report
        "-t", # Generate PyMOL .pse session file
    ], capture_output=True, text=True)
    # control
    if result.returncode == 0:
        print(f"Results saved to {target_dir}")
    else:
        print(f"Error processing {pdb_name}:")
        print(result.stderr)

Results saved to plip_results/Nb.AQ100_Adiponectin_model_0


In [8]:
# extract interacting residues, residue numbers and residue types from plip xml report

import xml.etree.ElementTree as ET

def get_interacting_residues(xml_path, peptide_chain="C"):

    tree = ET.parse(xml_path)
    root = tree.getroot()

    interacting_res = set()

    interaction_groups = root.findall(".//interactions/*")

    if not interaction_groups:
        print(f"No interactions found in {xml_path}")
        return []

    for group in interaction_groups:
        for contact in group:

            res_chain = contact.findtext("reschain")
            res_num = contact.findtext("resnr")
            res_type = contact.findtext("restype")

            if not all([res_chain, res_num, res_type]):
                continue

            res_chain = res_chain.strip()
            res_num = res_num.strip()
            res_type = res_type.strip()

            # exclude peptide chain
            if res_chain == peptide_chain:
                continue

            interacting_res.add((res_chain, res_num, res_type))

    # sort numerically if possible
    def sort_key(x):
        chain, num, _ = x
        try:
            return (chain, int(num))
        except ValueError:
            return (chain, num)

    return sorted(interacting_res, key=sort_key)

In [10]:
# test
residues = get_interacting_residues("/home/bri/pymol/data/3ab/plip_results/Nb.AQ100_Adiponectin_mono/Nb.AQ100_Adiponectin_mono_report.xml")
print(residues) # (reschain, resnr, restype)

[('A', '29', 'PHE'), ('A', '30', 'ARG'), ('A', '33', 'PHE'), ('A', '52', 'THR'), ('A', '56', 'ILE'), ('A', '58', 'ASN'), ('A', '99', 'ILE'), ('A', '100', 'LEU'), ('A', '101', 'VAL'), ('A', '103', 'VAL'), ('A', '105', 'GLY'), ('A', '107', 'TYR'), ('A', '110', 'ARG')]
